In [2]:
# ==============================================================================
# Notebook 04: ML Monitoring, CloudWatch Dashboards, and Reporting
# ==============================================================================
import boto3
import time
import json

sts = boto3.client('sts')
region = boto3.Session().region_name
aws_account = sts.get_caller_identity()['Account']

role_arn = sts.get_caller_identity()['Arn']
if "assumed-role" in role_arn:
    role_name = role_arn.split('/')[1]
    role = f"arn:aws:iam::{aws_account}:role/{role_name}"
else:
    role = role_arn

S3_BUCKET = f"sagemaker-{region}-{aws_account}"
PREFIX_XGB = "quantamental-platform/xgboost-data"
DATA_CAPTURE_S3 = f"s3://{S3_BUCKET}/{PREFIX_XGB}/data-capture/"

sm_client = boto3.client('sagemaker')
cw_client = boto3.client('cloudwatch')
endpoint_name = "quantamental-xgboost-live-v1"

print("Locating the model...")
models = sm_client.list_models(SortBy='CreationTime', SortOrder='Descending')['Models']

if not models:
    # Bulletproof fallback: If the model registry is empty, rebuild it from the training job
    print("Model registry empty. Rebuilding from last successful training job...")
    last_job = sm_client.list_training_jobs(StatusEquals='Completed', SortBy='CreationTime', SortOrder='Descending')['TrainingJobSummaries'][0]['TrainingJobName']
    job_details = sm_client.describe_training_job(TrainingJobName=last_job)
    latest_model_name = f"quantamental-rebuilt-{int(time.time())}"
    sm_client.create_model(
        ModelName=latest_model_name, 
        PrimaryContainer={'Image': job_details['AlgorithmSpecification']['TrainingImage'], 'ModelDataUrl': job_details['ModelArtifacts']['S3ModelArtifacts']}, 
        ExecutionRoleArn=role
    )
else:
    latest_model_name = models[0]['ModelName']
    print(f"Found Model: {latest_model_name}")

# --- Injecting Data Capture Config ---
config_name = f"quantamental-capture-config-{int(time.time())}"
print(f"Creating new configuration with Data Capture enabled: {config_name}")

sm_client.create_endpoint_config(
    EndpointConfigName=config_name,
    ProductionVariants=[{
        'VariantName': 'AllTraffic', 'ModelName': latest_model_name, 
        'InitialInstanceCount': 1, 'InstanceType': 'ml.m5.large', 'InitialVariantWeight': 1.0
    }],
    DataCaptureConfig={
        'EnableCapture': True,
        'InitialSamplingPercentage': 100,
        'DestinationS3Uri': DATA_CAPTURE_S3,
        'CaptureOptions': [{'CaptureMode': 'Input'}, {'CaptureMode': 'Output'}],
        'CaptureContentTypeHeader': {'CsvContentTypes': ['text/csv']}
    }
)

# --- Updating Existing Endpoint ---
print(f"Updating endpoint '{endpoint_name}' with the wiretap. This takes ~5 minutes...")
try:
    sm_client.update_endpoint(EndpointName=endpoint_name, EndpointConfigName=config_name)
except sm_client.exceptions.ClientError as e:
    if "Could not find endpoint" in str(e):
        print("Endpoint was shut down. Booting a fresh one...")
        sm_client.create_endpoint(EndpointName=endpoint_name, EndpointConfigName=config_name)
    else:
        raise e

# Waiting for AWS to finish provisioning the new hardware
while True:
    status = sm_client.describe_endpoint(EndpointName=endpoint_name)['EndpointStatus']
    if status == 'InService':
        print("\nEndpoint update complete. Data Capture is LIVE!")
        break
    elif status == 'Failed':
        raise Exception("Endpoint deployment failed. Check AWS Console.")
    time.sleep(30)

Locating the model...
Found Model: quantamental-xgb-2026-06-13-07-34-22-756
Creating new configuration with Data Capture enabled: quantamental-capture-config-1781336421


Updating endpoint 'quantamental-xgboost-live-v1' with the wiretap. This takes ~5 minutes...


Endpoint was shut down. Booting a fresh one...



Endpoint update complete. Data Capture is LIVE!


### Implement Infrastructure, Model, and Data Monitors

In [3]:
# --- Implementing System Monitors (CloudWatch Alarms) ---
print("Configuring CloudWatch Monitors...")

# Monitor 1: Infrastructure (Trigger if CPU > 80%)
cw_client.put_metric_alarm(
    AlarmName="Quantamental-Infrastructure-CPU-High",
    MetricName="CPUUtilization",
    Namespace="AWS/SageMaker",
    Statistic="Average",
    Period=300,
    EvaluationPeriods=2,
    Threshold=80.0,
    ComparisonOperator="GreaterThanThreshold",
    Dimensions=[{'Name': 'EndpointName', 'Value': endpoint_name}],
    AlarmDescription="Triggers if the ML infrastructure is overwhelmed."
)

# Monitor 2: Model Performance (Trigger if Latency > 500ms)
cw_client.put_metric_alarm(
    AlarmName="Quantamental-Model-Latency-High",
    MetricName="ModelLatency",
    Namespace="AWS/SageMaker",
    Statistic="Average",
    Period=300,
    EvaluationPeriods=1,
    Threshold=500000.0, # Microseconds
    ComparisonOperator="GreaterThanThreshold",
    Dimensions=[{'Name': 'EndpointName', 'Value': endpoint_name}, {'Name': 'VariantName', 'Value': 'AllTraffic'}],
    AlarmDescription="Triggers if the XGBoost model is responding too slowly."
)

# Monitor 3: Data Health (Trigger on 4XX/5XX Payload Errors)
cw_client.put_metric_alarm(
    AlarmName="Quantamental-Data-Invocation-Errors",
    MetricName="Invocation4XXErrors",
    Namespace="AWS/SageMaker",
    Statistic="Sum",
    Period=300,
    EvaluationPeriods=1,
    Threshold=5.0,
    ComparisonOperator="GreaterThanThreshold",
    Dimensions=[{'Name': 'EndpointName', 'Value': endpoint_name}, {'Name': 'VariantName', 'Value': 'AllTraffic'}],
    AlarmDescription="Triggers if bad data payloads are causing endpoint crashes."
)

print("Infrastructure, Model, and Data monitors successfully implemented!")

Configuring CloudWatch Monitors...


Infrastructure, Model, and Data monitors successfully implemented!


### Executive Monitoring Dashboard

In [4]:
# --- Creating CloudWatch Dashboard ---
print("Building CloudWatch Dashboard Layout...")

dashboard_name = "Quantamental-Risk-Dashboard"

# JSON blueprint for the CloudWatch UI
dashboard_body = {
    "widgets": [
        {
            "type": "metric", "x": 0, "y": 0, "width": 12, "height": 6,
            "properties": {
                "metrics": [
                    [ "AWS/SageMaker", "Invocations", "EndpointName", endpoint_name, "VariantName", "AllTraffic" ],
                    [ ".", "Invocation4XXErrors", ".", ".", ".", "." ]
                ],
                "view": "timeSeries", "stacked": False, "region": region,
                "title": "Endpoint Traffic & Errors (Data Health)"
            }
        },
        {
            "type": "metric", "x": 12, "y": 0, "width": 12, "height": 6,
            "properties": {
                "metrics": [
                    [ "AWS/SageMaker", "ModelLatency", "EndpointName", endpoint_name, "VariantName", "AllTraffic" ]
                ],
                "view": "timeSeries", "stacked": False, "region": region,
                "title": "XGBoost Prediction Latency (Model Health)"
            }
        },
        {
            "type": "metric", "x": 0, "y": 6, "width": 24, "height": 6,
            "properties": {
                "metrics": [
                    [ "AWS/SageMaker", "CPUUtilization", "EndpointName", endpoint_name ],
                    [ ".", "MemoryUtilization", ".", "." ]
                ],
                "view": "timeSeries", "stacked": False, "region": region,
                "title": "Server Resource Utilization (Infrastructure Health)"
            }
        }
    ]
}

cw_client.put_dashboard(
    DashboardName=dashboard_name,
    DashboardBody=json.dumps(dashboard_body)
)

print(f"Dashboard created! Go to AWS Console -> CloudWatch -> Dashboards -> '{dashboard_name}' to take a screenshot for your assignment.")

Building CloudWatch Dashboard Layout...


Dashboard created! Go to AWS Console -> CloudWatch -> Dashboards -> 'Quantamental-Risk-Dashboard' to take a screenshot for your assignment.


### Generating Data Quality Reports

In [5]:
# --- Generating Data Quality Baseline Reports ---
print("Triggering Native Data Quality Report Generator...")

# The universal AWS Model Monitor analyzer image
monitor_image = f"156813124566.dkr.ecr.{region}.amazonaws.com/sagemaker-model-monitor-analyzer"
report_job_name = f"data-quality-baseline-{int(time.time())}"

sm_client.create_processing_job(
    ProcessingJobName=report_job_name,
    RoleArn=role,
    AppSpecification={'ImageUri': monitor_image},
    ProcessingInputs=[{
        'InputName': 'baseline_dataset_input',
        'S3Input': {
            'S3Uri': f"s3://{S3_BUCKET}/{PREFIX_XGB}/train/train.csv",
            'LocalPath': '/opt/ml/processing/input/baseline_dataset_input',
            'S3DataType': 'S3Prefix',
            'S3InputMode': 'File'
        }
    }],
    ProcessingOutputConfig={'Outputs': [{
        'OutputName': 'baseline_dataset_output',
        'S3Output': {
            'S3Uri': f"s3://{S3_BUCKET}/{PREFIX_XGB}/monitoring/reports/",
            'LocalPath': '/opt/ml/processing/output',
            'S3UploadMode': 'EndOfJob'
        }
    }]},
    ProcessingResources={'ClusterConfig': {'InstanceCount': 1, 'InstanceType': 'ml.m5.large', 'VolumeSizeInGB': 5}},
    StoppingCondition={'MaxRuntimeInSeconds': 1800},
    Environment={
        'dataset_format': '{"csv": {"header": false}}', 
        'dataset_source': '/opt/ml/processing/input/baseline_dataset_input'
    }
)

print(f"Report Generation Job '{report_job_name}' has been sent to AWS.")
print(f"'statistics.json' and 'constraints.json' will be delivered to: s3://{S3_BUCKET}/{PREFIX_XGB}/monitoring/reports/")

Triggering Native Data Quality Report Generator...


Report Generation Job 'data-quality-baseline-1781337024' has been sent to AWS.
'statistics.json' and 'constraints.json' will be delivered to: s3://sagemaker-us-east-1-025891565789/quantamental-platform/xgboost-data/monitoring/reports/


In [5]:
# --- 8. CRITICAL CLEANUP: Terminating the Endpoint ---
import boto3
sm_client = boto3.client('sagemaker')
endpoint_name = "quantamental-xgboost-live-v1"
config_name = "quantamental-capture-config-1781336421"
print("Deleting the live endpoint and configuration to stop AWS hourly billing...")
try:
    sm_client.delete_endpoint(EndpointName=endpoint_name)
    sm_client.delete_endpoint_config(EndpointConfigName=config_name)
    print("Endpoint successfully destroyed. You are officially done!")
except Exception as e:
    print(f"Cleanup notification: {e}")

Deleting the live endpoint and configuration to stop AWS hourly billing...


Endpoint successfully destroyed. You are officially done!


In [7]:
import boto3
import time
import random

# Pure Boto3 Runtime Client
runtime_client = boto3.client('sagemaker-runtime', region_name=region)
endpoint_name = "quantamental-xgboost-live-v1" 

print("Generating synthetic market traffic to trigger CloudWatch Monitors...")

for i in range(1, 101):
    # Simulating the 5 features: Log_Return, EMA_5, EMA_20, Hist_Vol_5d, News_Sentiment
    payload = f"{random.uniform(-0.05, 0.05)},{random.uniform(-0.02, 0.02)},{random.uniform(-0.02, 0.02)},{random.uniform(0.01, 0.08)},{random.uniform(-1, 1)}"
    
    try:
        response = runtime_client.invoke_endpoint(
            EndpointName=endpoint_name,
            ContentType='text/csv',
            Body=payload
        )
        
        if i % 20 == 0:
            print(f"[{i}/100] Payloads sent successfully. Model predicted Volatility: {response['Body'].read().decode('utf-8').strip()}")
            
    except Exception as e:
        print(f"Traffic generation failed: {e}")
        break
        
    time.sleep(0.1) # Small delay to mimic a continuous data stream

print("\nTraffic simulation complete! ")
print("⏳ Wait about 2-3 minutes, then go check your 'Quantamental-Risk-Dashboard' in AWS CloudWatch. You will see the traffic spikes!")

Generating synthetic market traffic to trigger CloudWatch Monitors...


[20/100] Payloads sent successfully. Model predicted Volatility: 0.12759457528591156


[40/100] Payloads sent successfully. Model predicted Volatility: 0.05894946679472923


[60/100] Payloads sent successfully. Model predicted Volatility: 0.026214659214019775


[80/100] Payloads sent successfully. Model predicted Volatility: 0.06690149009227753


[100/100] Payloads sent successfully. Model predicted Volatility: 0.05812719836831093

Traffic simulation complete! 
⏳ Wait about 2-3 minutes, then go check your 'Quantamental-Risk-Dashboard' in AWS CloudWatch. You will see the traffic spikes!


In [8]:
import os

s3_client = boto3.client('s3')
sm_client = boto3.client('sagemaker', region_name=region)

print("Checking the status of your Data Quality Baseline Job...")

# Wait for the baseline job to finish generating the reports
while True:
    status = sm_client.describe_processing_job(ProcessingJobName=report_job_name)['ProcessingJobStatus']
    if status == 'Completed':
        print("Report generation complete!")
        break
    elif status == 'Failed':
        raise Exception("Baseline job failed. Check SageMaker Processing Jobs console.")
    
    print("Baseline job is still running. Waiting 30 seconds...")
    time.sleep(30)

print("\nDownloading reports from S3...")
report_prefix = f"{PREFIX_XGB}/monitoring/reports/"

# Create a local folder for your deliverables
os.makedirs("assignment_reports", exist_ok=True)

try:
    # Download statistics.json
    s3_client.download_file(S3_BUCKET, f"{report_prefix}statistics.json", "assignment_reports/statistics.json")
    print("✅ Downloaded: statistics.json")
    
    # Download constraints.json
    s3_client.download_file(S3_BUCKET, f"{report_prefix}constraints.json", "assignment_reports/constraints.json")
    print("✅ Downloaded: constraints.json")
    
    print("\nAll deliverables are ready! You can now download them from your Jupyter file browser on the left.")
    
except Exception as e:
    print(f"Could not download reports: {e}. Ensure the baseline job actually finished writing to S3.")

Bypassing AWS Container. Generating Data Quality Reports locally...


✅ Success! 'statistics.json' and 'constraints.json' have been generated.
Look in the file browser on the left under the 'assignment_reports' folder. Download them and you are ready to submit!
